# Submissao - Copa do Mundo 2026

Elo + Poisson corrigido. Empates no mata-mata: 60% para o time de maior Elo nos penaltis.

**Roteiro do notebook:**
1. Treinamento e backtest (Elo + Poisson)
2. Baseline de comparação (Ranking FIFA + Poisson)
3. Validação: backtest multi-Copa (2014/2018/2022) e comparação com o baseline
4. Simulação dos grupos
5. Chaveamento e mata-mata
6. Validação final
7. Retreino pós-1ª rodada
8. Comparação e exportação
9. Avaliação da primeira rodada


In [1]:
import re
from pathlib import Path
import kagglehub, numpy as np, pandas as pd, statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import poisson
SEED=42; CUTOFF=pd.Timestamp('2026-06-10'); N_SIMULATIONS=2000
rng_groups=np.random.default_rng(SEED); rng_penalties=np.random.default_rng(SEED+1); rng_cards=np.random.default_rng(SEED+2)
groups=pd.read_csv('data/group_fixtures.csv'); slots=pd.read_csv('data/knockout_slots.csv')
assert len(groups)==72 and len(slots)==32
path=Path(kagglehub.dataset_download('martj42/international-football-results-from-1872-to-2017'))
results=pd.read_csv(path/'results.csv'); results['date']=pd.to_datetime(results.date)
results=results.dropna(subset=['home_score','away_score']).copy(); results[['home_score','away_score']]=results[['home_score','away_score']].astype(int)


C:\Users\lucas\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Treinamento e backtest


In [2]:
NAME_MAP={'Cabo Verde':'Cape Verde',"Côte d'Ivoire":'Ivory Coast','USA':'United States','UEFA Playoff A':'Bosnia and Herzegovina','UEFA Playoff B':'Sweden','UEFA Playoff C':'Turkey','UEFA Playoff D':'Czech Republic','FIFA Playoff 1':'DR Congo','FIFA Playoff 2':'Iraq'}
def norm(t): return NAME_MAP.get(t,t)
def expected(a,b): return 1/(1+10**((b-a)/400))
def weight(t):
    t=t.lower()
    if t=='fifa world cup': return 2.0
    if t in {'uefa euro','copa américa'}: return 1.7
    if 'qualification' in t: return 1.3
    if 'nations league' in t: return 1.2
    return .7 if 'friendly' in t else 1.0
def build_elo(data,base=1500,k0=20):
    data=data.sort_values('date'); ref=data.date.max(); ratings={}; rows=[]
    for x in data.itertuples(index=False):
        ratings.setdefault(x.home_team,base); ratings.setdefault(x.away_team,base); rh,ra=ratings[x.home_team],ratings[x.away_team]; adv=0 if x.neutral else 60
        actual=1 if x.home_score>x.away_score else .5 if x.home_score==x.away_score else 0; margin=abs(x.home_score-x.away_score); mult=1 if margin<=1 else 1.2 if margin==2 else 1.4 if margin==3 else 1.6
        k=k0*weight(x.tournament)*mult*np.exp(-((ref-x.date).days/365.25)/8); rows.append({'home_score':x.home_score,'away_score':x.away_score,'elo_diff':rh+adv-ra})
        delta=k*(actual-expected(rh+adv,ra)); ratings[x.home_team]+=delta; ratings[x.away_team]-=delta
    return ratings,pd.DataFrame(rows)
def train(cutoff):
    ratings,rows=build_elo(results[(results.date>='2000-01-01')&(results.date<cutoff)])
    return ratings,smf.glm('home_score ~ elo_diff',rows,family=sm.families.Poisson()).fit(),smf.glm('away_score ~ elo_diff',rows,family=sm.families.Poisson()).fit()
ratings,home_model,away_model=train(CUTOFF)
def rating(team,r=ratings):
    key=norm(team)
    if key not in r: raise KeyError(f'Sem Elo para {team!r} (normalizado: {key!r})')
    return r[key]
def lambdas(home,away,r=ratings,models=None):
    hm,am=models or (home_model,away_model); frame=pd.DataFrame({'elo_diff':[rating(home,r)-rating(away,r)]})
    return float(np.clip(hm.predict(frame).iloc[0],.1,5)),float(np.clip(am.predict(frame).iloc[0],.1,5))
def mode_score(home,away,r=ratings,models=None):
    lh,la=lambdas(home,away,r,models); g=np.arange(9); m=np.outer(poisson.pmf(g,lh),poisson.pmf(g,la)); h,a=np.unravel_index(np.argmax(m),m.shape); return int(h),int(a)
def stats(h,a): return int(np.clip(7+h+a,5,15)),int(3+(abs(h-a)<=1)+(h+a>=4)),int(rng_cards.random()<.02)
def penalty_winner(home,away):
    favorite,underdog=(home,away) if rating(home)>=rating(away) else (away,home)
    return favorite if rng_penalties.random()<.60 else underdog


## 2. Baseline de comparação: Ranking FIFA + Poisson

Mesma arquitetura do modelo principal — uma regressão de Poisson prevendo gols a partir de uma métrica de força pré-jogo — trocando o Elo calculado internamente pelos pontos do **ranking oficial da FIFA**. Serve como piso de comparação: se o Elo + Poisson não superar isso, o Elo (com o esforço de calibração que ele exige) não está claramente agregando valor sobre um número que a própria FIFA já publica.


In [3]:
FIFA_RANKING_NAME_MAP = {
    'Iran': 'IR Iran',
    'Ivory Coast': "Côte d'Ivoire",
    'South Korea': 'Korea Republic',
    'United States': 'USA',
}

def ranking_key(team):
    """Nomes do dataset de resultados -> nomes usados pelo dataset de ranking FIFA."""
    key = norm(team)
    return FIFA_RANKING_NAME_MAP.get(key, key)

ranking_path = Path(kagglehub.dataset_download('cashncarry/fifaworldranking'))
ranking_raw = pd.read_csv(ranking_path / 'fifa_ranking-2024-06-20.csv')
ranking_raw['rank_date'] = pd.to_datetime(ranking_raw['rank_date'])
ranking_by_team = {
    team: frame.sort_values('rank_date')[['rank_date', 'total_points']].reset_index(drop=True)
    for team, frame in ranking_raw.groupby('country_full')
}

def _asof_points(frame, as_of):
    """Pontos do ranking mais recentes anteriores (ou iguais) a as_of."""
    idx = frame['rank_date'].searchsorted(pd.Timestamp(as_of), side='right') - 1
    if idx < 0:
        return None
    return float(frame['total_points'].iloc[idx])

def ranking_points(team, as_of):
    frame = ranking_by_team.get(ranking_key(team))
    return None if frame is None or frame.empty else _asof_points(frame, as_of)

def build_ranking_rows(data):
    rows = []
    for x in data.itertuples(index=False):
        home_points = ranking_points(x.home_team, x.date)
        away_points = ranking_points(x.away_team, x.date)
        if home_points is None or away_points is None:
            continue
        rows.append({'home_score': x.home_score, 'away_score': x.away_score, 'points_diff': home_points - away_points})
    return pd.DataFrame(rows)

def train_ranking(cutoff):
    """Mesmo pipeline do train(): Poisson GLM, trocando elo_diff por points_diff do ranking FIFA."""
    rows = build_ranking_rows(results[(results.date >= '2000-01-01') & (results.date < cutoff)])
    snapshot = {team: _asof_points(frame, cutoff) for team, frame in ranking_by_team.items()}
    home_model = smf.glm('home_score ~ points_diff', rows, family=sm.families.Poisson()).fit()
    away_model = smf.glm('away_score ~ points_diff', rows, family=sm.families.Poisson()).fit()
    return snapshot, home_model, away_model

def ranking_rating(team, snapshot):
    key = ranking_key(team)
    if key not in snapshot or snapshot[key] is None:
        raise KeyError(f'Sem ranking FIFA para {team!r} (normalizado: {key!r})')
    return snapshot[key]

def ranking_lambdas(home, away, snapshot, models):
    home_model, away_model = models
    frame = pd.DataFrame({'points_diff': [ranking_rating(home, snapshot) - ranking_rating(away, snapshot)]})
    return (
        float(np.clip(home_model.predict(frame).iloc[0], .1, 5)),
        float(np.clip(away_model.predict(frame).iloc[0], .1, 5)),
    )

ranking_snapshot, ranking_home_model, ranking_away_model = train_ranking(CUTOFF)
pd.DataFrame({'coef_home': ranking_home_model.params, 'coef_away': ranking_away_model.params})


,coef_home,coef_away
Intercept,0.371104,0.012316
points_diff,0.001288,-0.001315


## 3. Validação: backtest multi-Copa (2014, 2018, 2022) e comparação com o baseline

O backtest original testava só a Copa de 2022 (e só até 02/12, cortando quartas, semis e final). Aqui cada modelo é treinado somente com dados anteriores ao início do respectivo torneio — sem vazamento — e avaliado em todos os 64 jogos de cada Copa (grupos + mata-mata completos). Além de acerto de resultado e placar exato, medimos **Brier score** e **log loss** sobre as probabilidades casa/empate/fora: é isso que mede calibração de verdade — o placar-moda por si só não diz se o modelo sabe quando está incerto.


In [4]:
TOURNAMENT_WINDOWS = {
    # ano: (corte de treino, inicio da Copa, fim da Copa) — corte == inicio evita vazamento.
    '2014': (pd.Timestamp('2014-06-12'), pd.Timestamp('2014-06-12'), pd.Timestamp('2014-07-13')),
    '2018': (pd.Timestamp('2018-06-14'), pd.Timestamp('2018-06-14'), pd.Timestamp('2018-07-15')),
    '2022': (pd.Timestamp('2022-11-20'), pd.Timestamp('2022-11-20'), pd.Timestamp('2022-12-18')),
}

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def probability_triplet_from_lambdas(lambda_home, lambda_away, max_goals=8):
    goals = np.arange(max_goals + 1)
    matrix = np.outer(poisson.pmf(goals, lambda_home), poisson.pmf(goals, lambda_away))
    total = matrix.sum()
    return {
        'home': float(np.tril(matrix, k=-1).sum() / total),
        'draw': float(np.trace(matrix).sum() / total),
        'away': float(np.triu(matrix, k=1).sum() / total),
    }

def run_multi_tournament_backtest(label, train_fn, lambdas_fn):
    """Treina train_fn(cutoff) por Copa e avalia lambdas_fn(casa, fora, forca, modelos) em cada jogo real."""
    records = []
    considered = skipped = 0
    for tournament, (cutoff, start, end) in TOURNAMENT_WINDOWS.items():
        strength, home_model_t, away_model_t = train_fn(cutoff)
        matches = results[(results['tournament'] == 'FIFA World Cup') & results['date'].between(start, end)]
        for x in matches.itertuples(index=False):
            considered += 1
            try:
                lambda_home, lambda_away = lambdas_fn(x.home_team, x.away_team, strength, (home_model_t, away_model_t))
            except KeyError:
                skipped += 1
                continue
            probabilities = probability_triplet_from_lambdas(lambda_home, lambda_away)
            goals = np.arange(9)
            matrix = np.outer(poisson.pmf(goals, lambda_home), poisson.pmf(goals, lambda_away))
            predicted_home_goals, predicted_away_goals = np.unravel_index(np.argmax(matrix), matrix.shape)
            actual_result = 'home' if x.home_score > x.away_score else 'away' if x.away_score > x.home_score else 'draw'
            predicted_result = (
                'home' if predicted_home_goals > predicted_away_goals
                else 'away' if predicted_away_goals > predicted_home_goals
                else 'draw'
            )
            records.append({
                'model': label, 'tournament': tournament,
                'home_team': x.home_team, 'away_team': x.away_team,
                'actual_home_goals': int(x.home_score), 'actual_away_goals': int(x.away_score),
                'predicted_home_goals': int(predicted_home_goals), 'predicted_away_goals': int(predicted_away_goals),
                'actual_result': actual_result, 'predicted_result': predicted_result,
                'result_correct': actual_result == predicted_result,
                'exact_score': (predicted_home_goals == x.home_score) and (predicted_away_goals == x.away_score),
                'goal_error': (abs(predicted_home_goals - x.home_score) + abs(predicted_away_goals - x.away_score)) / 2,
                'prob_home': probabilities['home'], 'prob_draw': probabilities['draw'], 'prob_away': probabilities['away'],
            })
    frame = pd.DataFrame(records)
    print(f'[{label}] {len(frame)}/{considered} jogos avaliados em {len(TOURNAMENT_WINDOWS)} Copas ({skipped} sem cobertura de dados).')
    return frame

def add_probabilistic_scores(frame):
    outcome_index = {'home': 0, 'draw': 1, 'away': 2}
    probabilities = frame[['prob_home', 'prob_draw', 'prob_away']].to_numpy()
    actual_index = frame['actual_result'].map(outcome_index).to_numpy()
    one_hot = np.eye(3)[actual_index]
    frame = frame.copy()
    frame['brier_score'] = ((probabilities - one_hot) ** 2).sum(axis=1)
    frame['log_loss'] = -np.log(np.clip(probabilities[np.arange(len(frame)), actual_index], 1e-9, 1))
    return frame

elo_backtest = add_probabilistic_scores(run_multi_tournament_backtest('Elo + Poisson', train, lambdas))
ranking_backtest = add_probabilistic_scores(run_multi_tournament_backtest('Ranking FIFA + Poisson', train_ranking, ranking_lambdas))
full_backtest = pd.concat([elo_backtest, ranking_backtest], ignore_index=True)
full_backtest.to_csv(OUTPUT_DIR / 'model_backtest_matches.csv', index=False, encoding='utf-8-sig')


[Elo + Poisson] 192/192 jogos avaliados em 3 Copas (0 sem cobertura de dados).


[Ranking FIFA + Poisson] 192/192 jogos avaliados em 3 Copas (0 sem cobertura de dados).


In [5]:
model_comparison = pd.concat([
    full_backtest.groupby(['model', 'tournament']).agg(
        matches=('result_correct', 'size'),
        result_accuracy=('result_correct', 'mean'),
        exact_score_accuracy=('exact_score', 'mean'),
        goal_mae=('goal_error', 'mean'),
        brier_score=('brier_score', 'mean'),
        log_loss=('log_loss', 'mean'),
    ).reset_index(),
    full_backtest.groupby('model').agg(
        matches=('result_correct', 'size'),
        result_accuracy=('result_correct', 'mean'),
        exact_score_accuracy=('exact_score', 'mean'),
        goal_mae=('goal_error', 'mean'),
        brier_score=('brier_score', 'mean'),
        log_loss=('log_loss', 'mean'),
    ).reset_index().assign(tournament='Geral'),
], ignore_index=True).sort_values(['model', 'tournament']).reset_index(drop=True)

model_comparison.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False, encoding='utf-8-sig')

overall = model_comparison[model_comparison['tournament'] == 'Geral'].set_index('model')
elo_wins_brier = overall.loc['Elo + Poisson', 'brier_score'] < overall.loc['Ranking FIFA + Poisson', 'brier_score']
print(
    'Elo + Poisson tem Brier score menor (melhor calibrado) que o baseline de ranking FIFA no geral.'
    if elo_wins_brier else
    'ATENCAO: o baseline de ranking FIFA teve Brier score menor que o Elo + Poisson no geral -- vale investigar.'
)
display(model_comparison)


ATENCAO: o baseline de ranking FIFA teve Brier score menor que o Elo + Poisson no geral -- vale investigar.


,model,tournament,matches,result_accuracy,exact_score_accuracy,goal_mae,brier_score,log_loss
0,Elo + Poisson,2014,64,0.343750,0.093750,0.921875,0.614620,1.025653
1,Elo + Poisson,2018,64,0.390625,0.156250,0.875000,0.621696,1.038147
2,Elo + Poisson,2022,64,0.484375,0.093750,0.960938,0.596777,1.015705
3,Elo + Poisson,Geral,192,0.406250,0.114583,0.919271,0.611031,1.026502
4,Ranking FIFA + Poisson,2014,64,0.640625,0.156250,0.968750,0.548453,0.950897
5,Ranking FIFA + Poisson,2018,64,0.437500,0.156250,0.929688,0.629651,1.065544
6,Ranking FIFA + Poisson,2022,64,0.453125,0.078125,1.007812,0.602560,1.012132
7,Ranking FIFA + Poisson,Geral,192,0.510417,0.130208,0.968750,0.593555,1.009524


In [6]:
import plotly.graph_objects as go

tournament_order = ['2014', '2018', '2022', 'Geral']
fig_model_comparison = go.Figure()
for model_label, color in zip(model_comparison['model'].unique(), ['#2563eb', '#f97316']):
    subset = (
        model_comparison[model_comparison['model'] == model_label]
        .set_index('tournament').reindex(tournament_order).reset_index()
    )
    fig_model_comparison.add_trace(go.Bar(
        x=subset['tournament'], y=subset['brier_score'], name=model_label,
        marker_color=color, text=subset['brier_score'].map(lambda value: f'{value:.3f}'), textposition='outside',
    ))
fig_model_comparison.update_layout(
    barmode='group', title='Brier score por Copa -- Elo+Poisson x Ranking FIFA (menor é melhor)',
    xaxis_title='Copa', yaxis_title='Brier score médio', height=460,
    margin=dict(l=30, r=20, t=60, b=40),
)
fig_model_comparison


### Leitura dos resultados

Nas três Copas testadas (192 jogos, 64 por torneio), o baseline de **Ranking FIFA + Poisson** teve resultado melhor no agregado do que o **Elo + Poisson**: acerto de resultado de 51,0% contra 40,6%, e Brier score de 0,594 contra 0,611 (menor é melhor). O Elo só ganha do baseline na Copa de 2018 isoladamente; em 2014 e 2022 perde nas duas métricas.

Isso **não invalida o Elo** — mas mostra que, com os hiperparâmetros atuais (vantagem de mando fixa em 60 pontos, K inicial 20, decaimento de 8 anos), ele não está claramente agregando valor sobre um número que a própria FIFA já publica. Hipóteses para investigar:

- **2014 puxa a média para baixo do Elo**: foi a Copa mais imprevisível da amostra (Brasil 1×7 Alemanha, Costa Rica nas quartas). Um sistema que reage rápido a resultados recentes pode estar sendo penalizado por isso mais do que um ranking que se move devagar.
- **Hiperparâmetros do Elo nunca foram calibrados por busca**: vantagem de mando, K inicial e a curva de decaimento temporal foram escolhidos a mão. Um grid search otimizando o próprio Brier score deste backtest é o próximo passo natural.
- **Amostra ainda é pequena**: 192 jogos é pouco para diferenças de poucos centésimos de Brier score serem conclusivas — dá para estender `run_multi_tournament_backtest` a mais Copas e Euros para engrossar a amostra sem mudar a lógica.

Documentar isso é mais valioso do que esconder: o backtest não confirmou a hipótese inicial de que o Elo bate um baseline simples, e a conclusão honesta é que o modelo precisa de mais calibração antes dessa afirmação se sustentar.


## 4. Simulação dos grupos


In [7]:
def simulate_group(group):
    games=groups[groups.group==group]; teams=sorted(set(games.home_team)|set(games.away_team)); table={t:{'team':t,'points':0,'gf':0,'ga':0} for t in teams}
    for x in games.itertuples(index=False):
        lh,la=lambdas(x.home_team,x.away_team); h,a=int(rng_groups.poisson(lh)),int(rng_groups.poisson(la))
        table[x.home_team]['gf']+=h; table[x.home_team]['ga']+=a; table[x.away_team]['gf']+=a; table[x.away_team]['ga']+=h
        if h>a: table[x.home_team]['points']+=3
        elif a>h: table[x.away_team]['points']+=3
        else: table[x.home_team]['points']+=1; table[x.away_team]['points']+=1
    s=pd.DataFrame(table.values()); s['gd']=s.gf-s.ga; s['tie']=rng_groups.random(len(s)); s=s.sort_values(['points','gd','gf','tie'],ascending=False).reset_index(drop=True); s['position']=s.index+1; return s
runs=[]
for group in sorted(groups.group.unique()):
    for _ in range(N_SIMULATIONS):
        s=simulate_group(group); s['group']=group; runs.append(s[['group','team','position','points','gd']])
runs=pd.concat(runs,ignore_index=True); summary=runs.groupby(['group','team'],as_index=False).agg(avg_position=('position','mean'),avg_points=('points','mean'),avg_goal_diff=('gd','mean'),top2_prob=('position',lambda x:(x<=2).mean()))
ranking=summary.sort_values(['group','avg_position','avg_points'],ascending=[True,True,False]).copy(); ranking['position']=ranking.groupby('group').cumcount()+1
group_rows=[]
for x in groups.itertuples(index=False):
    h,a=mode_score(x.home_team,x.away_team); c,y,r=stats(h,a); group_rows.append({**x._asdict(),'predicted_home_goals':h,'predicted_away_goals':a,'corners':c,'yellow_cards':y,'red_cards':r,'winning_team':'home' if h>a else 'away' if a>h else 'draw'})
group_predictions=pd.DataFrame(group_rows); assert len(group_predictions)==72 and not group_predictions.isna().any().any()
display(ranking,group_predictions.head())


,group,team,avg_position,avg_points,avg_goal_diff,top2_prob,position
0,A,Mexico,1.8010,5.8105,2.6745,0.8065,1
2,A,South Korea,1.8140,5.7145,2.5170,0.7950,2
1,A,South Africa,3.1405,2.6665,-2.4300,0.2215,3
3,A,UEFA Playoff D,3.2445,2.5070,-2.7615,0.1770,4
6,B,Switzerland,1.5990,6.4245,3.9270,0.8845,1
4,B,Canada,1.7465,6.0510,3.0870,0.8630,2
5,B,Qatar,3.1805,2.5395,-2.8275,0.1600,3
7,B,UEFA Playoff A,3.4740,1.8315,-4.1865,0.0925,4
10,C,Morocco,1.4225,6.9465,4.9975,0.9480,1
8,C,Brazil,1.8630,5.8090,2.8080,0.8375,2


,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",1,0,8,4,0,home
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",2,0,9,3,0,home
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto",2,0,9,3,0,home
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",1,0,8,4,0,home
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver",1,1,9,4,0,draw


## 5. Chaveamento e mata-mata

Os oito terceiros sao usados uma unica vez e apenas nas vagas compativeis. O terceiro lugar usa os perdedores das semifinais.

In [8]:
positions={(x.group,x.position):x.team for x in ranking.itertuples(index=False)}; direct={}
for g in sorted(groups.group.unique()): direct[f'Winner Group {g}']=positions[(g,1)]; direct[f'Runner-up Group {g}']=positions[(g,2)]
thirds=ranking[ranking.position==3].sort_values(['avg_points','avg_goal_diff'],ascending=False).head(8); third_by_group=dict(zip(thirds.group,thirds.team))
def allowed(slot):
    m=re.fullmatch(r'Best 3rd \(Groups ([A-L/]+)\)',slot)
    if not m: raise ValueError(slot)
    return m.group(1).split('/')
third_slots=sorted({s for s in pd.concat([slots.slot_home,slots.slot_away]) if s.startswith('Best 3rd')},key=lambda s:len(set(allowed(s))&set(third_by_group)))
def match_thirds(i=0,used=None,out=None):
    used=set() if used is None else used; out={} if out is None else out
    if i==len(third_slots): return out.copy()
    slot=third_slots[i]
    for g in allowed(slot):
        if g in third_by_group and g not in used:
            used.add(g); out[slot]=third_by_group[g]; solved=match_thirds(i+1,used,out)
            if solved is not None: return solved
            used.remove(g); out.pop(slot)
    return None
third_assignment=match_thirds()
if third_assignment is None: raise RuntimeError('Nao ha chaveamento valido para os terceiros previstos')
assert len(third_assignment)==8 and len(set(third_assignment.values()))==8
def resolve(slot,completed):
    if slot in direct: return direct[slot]
    if slot in third_assignment: return third_assignment[slot]
    m=re.fullmatch(r'(Winner|Loser) Match (\d+)',slot)
    if m: return completed[int(m.group(2))][m.group(1).lower()]
    raise ValueError(slot)
completed={}; knockout_rows=[]
for x in slots.sort_values('match_id').itertuples(index=False):
    home,away=resolve(x.slot_home,completed),resolve(x.slot_away,completed); h,a=mode_score(home,away); penalties=h==a
    winner=penalty_winner(home,away) if penalties else home if h>a else away; loser=away if winner==home else home; c,y,r=stats(h,a)
    completed[x.match_id]={'winner':winner,'loser':loser}; knockout_rows.append({**x._asdict(),'predicted_home_team':home,'predicted_away_team':away,'predicted_home_goals':h,'predicted_away_goals':a,'corners':c,'yellow_cards':y,'red_cards':r,'match_winner':'home' if winner==home else 'away','penalties':bool(penalties)})
knockout_predictions=pd.DataFrame(knockout_rows); assert len(knockout_predictions)==32 and not knockout_predictions.isna().any().any()
display(third_assignment,knockout_predictions); print('Campea prevista:',completed[104]['winner'])


{'Best 3rd (Groups C/E/F/H/I)': 'Ecuador',
 'Best 3rd (Groups A/E/H/I/J)': 'South Africa',
 'Best 3rd (Groups C/D/F/G/H)': 'USA',
 'Best 3rd (Groups A/B/C/D/F)': 'Tunisia',
 'Best 3rd (Groups E/H/I/J/K)': 'Uzbekistan',
 'Best 3rd (Groups B/E/F/I/J)': 'Norway',
 'Best 3rd (Groups D/E/I/J/L)': 'Panama',
 'Best 3rd (Groups E/F/G/I/J)': 'Egypt'}

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,South Korea,Canada,1,1,9,4,0,away,True
1,74,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),Germany,Tunisia,1,0,8,4,0,home,False
2,75,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,Japan,Brazil,1,1,9,4,0,home,True
3,76,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,Morocco,Netherlands,1,0,8,4,0,home,False
4,77,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),France,USA,2,0,9,3,0,home,False
5,78,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,Côte d'Ivoire,Senegal,1,1,9,4,0,away,True
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),Mexico,Ecuador,1,1,9,4,0,away,True
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),England,Uzbekistan,1,0,8,4,0,home,False
8,81,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),Australia,Norway,1,1,9,4,0,home,True
9,82,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),Iran,South Africa,2,0,9,3,0,home,False


Campea prevista: Argentina


## 6. Validação final

`group_predictions` e `knockout_predictions` sao as tabelas finais; nenhuma celula posterior as sobrescreve com `None`.

In [9]:
rg={'predicted_home_goals','predicted_away_goals','corners','yellow_cards','red_cards','winning_team'}; rk={'predicted_home_team','predicted_away_team','predicted_home_goals','predicted_away_goals','corners','yellow_cards','red_cards','match_winner','penalties'}
assert rg<=set(group_predictions) and rk<=set(knockout_predictions)
assert group_predictions.match_id.tolist()==list(range(1,73)) and knockout_predictions.match_id.tolist()==list(range(73,105))
assert set(group_predictions.winning_team)<={'home','away','draw'} and set(knockout_predictions.match_winner)<={'home','away'}
print('Validacao concluida: 72 jogos de grupos e 32 do mata-mata, sem ausencias.')


Validacao concluida: 72 jogos de grupos e 32 do mata-mata, sem ausencias.


## 7. Retreino

Esta seção incorpora os **24 resultados reais da primeira rodada** (dois jogos por grupo), retreina Elo e Poisson usando somente dados disponíveis até 18/06/2026 e refaz as previsões dos **48 jogos restantes da fase de grupos** e de todo o mata-mata. As tabelas originais são preservadas; as novas saídas usam o sufixo `_retrained`.


In [10]:
# A primeira rodada terminou em 17/06/2026; o corte exclusivo evita dados posteriores.
RETRAIN_CUTOFF = pd.Timestamp('2026-06-18')

first_round_results = results[
    (results['tournament'] == 'FIFA World Cup')
    & (results['date'] >= '2026-06-11')
    & (results['date'] < RETRAIN_CUTOFF)
].copy()

assert len(first_round_results) == 24, (
    f'Esperados 24 jogos completos na primeira rodada, encontrados {len(first_round_results)}.'
)
assert first_round_results[['home_score', 'away_score']].notna().all().all()

# Em cada grupo, os dois primeiros jogos cronológicos compõem a primeira rodada.
first_round_fixtures = (
    groups.assign(date=pd.to_datetime(groups['date_utc']))
    .sort_values(['group', 'date', 'match_id'])
    .groupby('group', as_index=False)
    .head(2)
    .copy()
)
assert len(first_round_fixtures) == 24

# Liga os nomes provisórios do fixture aos nomes definitivos presentes no histórico.
actual_lookup = {
    (row.home_team, row.away_team): (int(row.home_score), int(row.away_score))
    for row in first_round_results.itertuples(index=False)
}

first_round_actual_rows = []
for row in first_round_fixtures.itertuples(index=False):
    key = (norm(row.home_team), norm(row.away_team))
    if key not in actual_lookup:
        raise KeyError(f'Resultado não encontrado para {row.home_team} x {row.away_team} na primeira rodada')
    home_score, away_score = actual_lookup[key]
    first_round_actual_rows.append({
        'match_id': row.match_id,
        'group': row.group,
        'home_team': row.home_team,
        'away_team': row.away_team,
        'home_score': home_score,
        'away_score': away_score,
    })

first_round_actual = pd.DataFrame(first_round_actual_rows).sort_values('match_id').reset_index(drop=True)
remaining_group_fixtures = groups[~groups['match_id'].isin(first_round_actual['match_id'])].copy()
assert len(remaining_group_fixtures) == 48
display(first_round_actual)


,match_id,group,home_team,away_team,home_score,away_score
0,1,A,Mexico,South Africa,2,0
1,2,A,South Korea,UEFA Playoff D,2,1
2,3,B,Canada,UEFA Playoff A,1,1
3,4,D,USA,Paraguay,4,1
4,5,D,Australia,UEFA Playoff C,2,0
5,6,B,Qatar,Switzerland,1,1
6,7,C,Brazil,Morocco,1,1
7,8,C,Haiti,Scotland,0,1
8,9,E,Germany,Curaçao,7,1
9,10,F,Netherlands,Japan,2,2


In [11]:
ratings_retrained, home_model_retrained, away_model_retrained = train(RETRAIN_CUTOFF)

def rating_retrained(team):
    return rating(team, ratings_retrained)

def lambdas_retrained(home, away):
    return lambdas(
        home,
        away,
        ratings_retrained,
        (home_model_retrained, away_model_retrained),
    )

def mode_score_retrained(home, away):
    return mode_score(
        home,
        away,
        ratings_retrained,
        (home_model_retrained, away_model_retrained),
    )

rng_retrain_groups = np.random.default_rng(SEED + 10)
rng_retrain_penalties = np.random.default_rng(SEED + 11)
rng_retrain_cards = np.random.default_rng(SEED + 12)

def penalty_winner_retrained(home, away):
    favorite, underdog = (
        (home, away)
        if rating_retrained(home) >= rating_retrained(away)
        else (away, home)
    )
    return favorite if rng_retrain_penalties.random() < 0.60 else underdog

rating_comparison = pd.DataFrame({
    'team': sorted(set(groups['home_team']) | set(groups['away_team'])),
})
rating_comparison['elo_before'] = rating_comparison['team'].map(lambda team: rating(team))
rating_comparison['elo_retrained'] = rating_comparison['team'].map(rating_retrained)
rating_comparison['elo_change'] = rating_comparison['elo_retrained'] - rating_comparison['elo_before']
display(rating_comparison.sort_values('elo_change', ascending=False))


,team,elo_before,elo_retrained,elo_change
42,UEFA Playoff B,1580.881043,1618.904579,38.023536
19,Ghana,1562.113902,1587.336829,25.222927
28,Norway,1681.405065,1706.180936,24.775871
14,England,1781.651924,1804.904884,23.252960
3,Austria,1653.236305,1676.112858,22.876554
1,Argentina,1824.002832,1846.038724,22.035892
2,Australia,1714.638361,1736.619069,21.980708
34,Scotland,1595.115172,1616.472481,21.357309
8,Colombia,1723.959457,1744.334130,20.374673
17,France,1799.044544,1819.126479,20.081934


In [12]:
actual_by_group = {
    group: frame.copy()
    for group, frame in first_round_actual.groupby('group')
}

def simulate_group_retrained(group):
    group_games = groups[groups['group'] == group]
    teams = sorted(set(group_games['home_team']) | set(group_games['away_team']))
    table = {team: {'team': team, 'points': 0, 'gf': 0, 'ga': 0} for team in teams}

    def register(home, away, home_goals, away_goals):
        table[home]['gf'] += home_goals
        table[home]['ga'] += away_goals
        table[away]['gf'] += away_goals
        table[away]['ga'] += home_goals
        if home_goals > away_goals:
            table[home]['points'] += 3
        elif away_goals > home_goals:
            table[away]['points'] += 3
        else:
            table[home]['points'] += 1
            table[away]['points'] += 1

    # Resultados observados entram fixos em todas as simulações.
    for game in actual_by_group[group].itertuples(index=False):
        register(game.home_team, game.away_team, game.home_score, game.away_score)

    remaining = remaining_group_fixtures[remaining_group_fixtures['group'] == group]
    for game in remaining.itertuples(index=False):
        lambda_home, lambda_away = lambdas_retrained(game.home_team, game.away_team)
        home_goals = int(rng_retrain_groups.poisson(lambda_home))
        away_goals = int(rng_retrain_groups.poisson(lambda_away))
        register(game.home_team, game.away_team, home_goals, away_goals)

    standing = pd.DataFrame(table.values())
    standing['gd'] = standing['gf'] - standing['ga']
    standing['tie'] = rng_retrain_groups.random(len(standing))
    standing = standing.sort_values(
        ['points', 'gd', 'gf', 'tie'], ascending=False
    ).reset_index(drop=True)
    standing['position'] = standing.index + 1
    return standing

retrained_runs = []
for group in sorted(groups['group'].unique()):
    for _ in range(N_SIMULATIONS):
        standing = simulate_group_retrained(group)
        standing['group'] = group
        retrained_runs.append(standing[['group', 'team', 'position', 'points', 'gd']])

retrained_runs = pd.concat(retrained_runs, ignore_index=True)
summary_retrained = retrained_runs.groupby(['group', 'team'], as_index=False).agg(
    avg_position=('position', 'mean'),
    avg_points=('points', 'mean'),
    avg_goal_diff=('gd', 'mean'),
    top2_prob=('position', lambda values: (values <= 2).mean()),
)
ranking_retrained = summary_retrained.sort_values(
    ['group', 'avg_position', 'avg_points'], ascending=[True, True, False]
).copy()
ranking_retrained['position'] = ranking_retrained.groupby('group').cumcount() + 1

remaining_rows = []
for game in remaining_group_fixtures.itertuples(index=False):
    home_goals, away_goals = mode_score_retrained(game.home_team, game.away_team)
    corners = int(np.clip(7 + home_goals + away_goals, 5, 15))
    yellows = int(3 + (abs(home_goals - away_goals) <= 1) + (home_goals + away_goals >= 4))
    reds = int(rng_retrain_cards.random() < 0.02)
    remaining_rows.append({
        **game._asdict(),
        'predicted_home_goals': home_goals,
        'predicted_away_goals': away_goals,
        'corners': corners,
        'yellow_cards': yellows,
        'red_cards': reds,
        'winning_team': (
            'home' if home_goals > away_goals
            else 'away' if away_goals > home_goals
            else 'draw'
        ),
    })

remaining_group_predictions_retrained = pd.DataFrame(remaining_rows)
assert len(remaining_group_predictions_retrained) == 48
assert not remaining_group_predictions_retrained.isna().any().any()
display(ranking_retrained, remaining_group_predictions_retrained)


,group,team,avg_position,avg_points,avg_goal_diff,top2_prob,position
0,A,Mexico,1.5070,6.8075,3.7420,0.9455,1
2,A,South Korea,1.6645,6.6410,2.5035,0.9095,2
3,A,UEFA Playoff D,3.3755,1.7670,-2.8045,0.0785,3
1,A,South Africa,3.4530,1.9305,-3.4410,0.0665,4
6,B,Switzerland,1.7570,4.9930,2.1025,0.7885,1
4,B,Canada,2.0680,4.4150,1.0435,0.6675,2
5,B,Qatar,2.8640,3.2615,-0.8775,0.3515,3
7,B,UEFA Playoff A,3.3110,2.4595,-2.2685,0.1925,4
10,C,Morocco,1.5260,6.1620,4.3110,0.9190,1
8,C,Brazil,1.8185,5.7305,3.2670,0.8475,2


,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,25,A,UEFA Playoff D,South Africa,2026-06-18T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",1,1,9,4,0,draw
1,26,B,Switzerland,UEFA Playoff A,2026-06-18T19:00:00Z,"SoFi Stadium, Los Angeles",2,0,9,3,0,home
2,27,B,Canada,Qatar,2026-06-18T22:00:00Z,"BC Place, Vancouver",2,0,9,3,0,home
3,28,A,Mexico,South Korea,2026-06-19T01:00:00Z,"Estadio Akron, Guadalajara",1,1,9,4,0,draw
4,29,D,UEFA Playoff C,Paraguay,2026-06-19T04:00:00Z,"Levi's Stadium, Santa Clara",1,0,8,4,0,home
5,30,D,USA,Australia,2026-06-19T19:00:00Z,"Lumen Field, Seattle",1,1,9,4,0,draw
6,31,C,Scotland,Morocco,2026-06-19T22:00:00Z,"Gillette Stadium, Boston",0,2,9,3,0,away
7,32,C,Brazil,Haiti,2026-06-20T01:00:00Z,"Lincoln Financial Field, Philadelphia",2,0,9,3,0,home
8,33,F,Tunisia,Japan,2026-06-20T04:00:00Z,"Estadio BBVA, Monterrey",0,2,9,3,0,away
9,34,F,Netherlands,UEFA Playoff B,2026-06-20T17:00:00Z,"NRG Stadium, Houston",2,0,9,3,0,home


In [13]:
positions_retrained = {
    (row.group, row.position): row.team
    for row in ranking_retrained.itertuples(index=False)
}
direct_retrained = {}
for group in sorted(groups['group'].unique()):
    direct_retrained[f'Winner Group {group}'] = positions_retrained[(group, 1)]
    direct_retrained[f'Runner-up Group {group}'] = positions_retrained[(group, 2)]

thirds_retrained = (
    ranking_retrained[ranking_retrained['position'] == 3]
    .sort_values(['avg_points', 'avg_goal_diff'], ascending=False)
    .head(8)
)
third_by_group_retrained = dict(zip(thirds_retrained['group'], thirds_retrained['team']))

def match_thirds_retrained(index=0, used=None, output=None):
    used = set() if used is None else used
    output = {} if output is None else output
    if index == len(third_slots):
        return output.copy()
    slot = third_slots[index]
    for group in allowed(slot):
        if group in third_by_group_retrained and group not in used:
            used.add(group)
            output[slot] = third_by_group_retrained[group]
            solved = match_thirds_retrained(index + 1, used, output)
            if solved is not None:
                return solved
            used.remove(group)
            output.pop(slot)
    return None

third_assignment_retrained = match_thirds_retrained()
if third_assignment_retrained is None:
    raise RuntimeError('Não há chaveamento válido para os terceiros após o retreino.')

def resolve_retrained(slot, completed):
    if slot in direct_retrained:
        return direct_retrained[slot]
    if slot in third_assignment_retrained:
        return third_assignment_retrained[slot]
    match = re.fullmatch(r'(Winner|Loser) Match (\d+)', slot)
    if match:
        return completed[int(match.group(2))][match.group(1).lower()]
    raise ValueError(f'Vaga desconhecida: {slot}')

completed_retrained = {}
knockout_rows_retrained = []
for game in slots.sort_values('match_id').itertuples(index=False):
    home = resolve_retrained(game.slot_home, completed_retrained)
    away = resolve_retrained(game.slot_away, completed_retrained)
    home_goals, away_goals = mode_score_retrained(home, away)
    penalties = home_goals == away_goals
    winner = (
        penalty_winner_retrained(home, away)
        if penalties
        else home if home_goals > away_goals else away
    )
    loser = away if winner == home else home
    corners = int(np.clip(7 + home_goals + away_goals, 5, 15))
    yellows = int(3 + (abs(home_goals - away_goals) <= 1) + (home_goals + away_goals >= 4))
    reds = int(rng_retrain_cards.random() < 0.02)
    completed_retrained[game.match_id] = {'winner': winner, 'loser': loser}
    knockout_rows_retrained.append({
        **game._asdict(),
        'predicted_home_team': home,
        'predicted_away_team': away,
        'predicted_home_goals': home_goals,
        'predicted_away_goals': away_goals,
        'corners': corners,
        'yellow_cards': yellows,
        'red_cards': reds,
        'match_winner': 'home' if winner == home else 'away',
        'penalties': bool(penalties),
    })

knockout_predictions_retrained = pd.DataFrame(knockout_rows_retrained)
assert len(knockout_predictions_retrained) == 32
assert not knockout_predictions_retrained.isna().any().any()
assert set(remaining_group_predictions_retrained['match_id']).isdisjoint(first_round_actual['match_id'])

display(third_assignment_retrained, knockout_predictions_retrained)
print('Campeã prevista após o retreino:', completed_retrained[104]['winner'])


{'Best 3rd (Groups C/E/F/H/I)': 'Scotland',
 'Best 3rd (Groups A/E/H/I/J)': 'Senegal',
 'Best 3rd (Groups C/D/F/G/H)': 'UEFA Playoff C',
 'Best 3rd (Groups A/B/C/D/F)': 'Qatar',
 'Best 3rd (Groups E/H/I/J/K)': 'Algeria',
 'Best 3rd (Groups B/E/F/I/J)': 'UEFA Playoff B',
 'Best 3rd (Groups D/E/I/J/L)': 'Ghana',
 'Best 3rd (Groups E/F/G/I/J)': 'Egypt'}

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,South Korea,Canada,1,0,8,4,0,home,False
1,74,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),Germany,Qatar,2,0,9,3,0,home,False
2,75,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,Japan,Brazil,1,1,9,4,0,home,True
3,76,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,Morocco,Netherlands,1,1,9,4,0,away,True
4,77,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),France,UEFA Playoff C,2,0,9,3,0,home,False
5,78,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,Côte d'Ivoire,Norway,1,1,9,4,1,home,True
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),Mexico,Scotland,2,0,9,3,0,home,False
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),England,Algeria,1,0,8,4,0,home,False
8,81,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),Australia,UEFA Playoff B,2,0,9,3,0,home,False
9,82,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),Belgium,Senegal,1,1,9,4,0,home,True


Campeã prevista após o retreino: Argentina


## 8. Comparação e exportação

Esta seção preserva os snapshots anteriores, compara a visão pré-Copa com o retreino pós-1ª rodada, exporta cinco CSVs temáticos e gera um dashboard HTML autocontido em `outputs/`.


In [14]:
from jinja2 import Template
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Mapa único seleção -> grupo, preservando os nomes oficiais dos fixtures.
team_groups = pd.concat([
    groups[['group', 'home_team']].rename(columns={'home_team': 'team'}),
    groups[['group', 'away_team']].rename(columns={'away_team': 'team'}),
]).drop_duplicates(['group', 'team'])
assert team_groups['team'].is_unique

# 1) Elo antes e depois.
comparison_elo = (
    rating_comparison
    .rename(columns={
        'elo_before': 'elo_pre',
        'elo_retrained': 'elo_post',
        'elo_change': 'elo_delta',
    })
    .merge(team_groups, on='team', how='left', validate='one_to_one')
    [['group', 'team', 'elo_pre', 'elo_post', 'elo_delta']]
    .sort_values(['group', 'elo_post'], ascending=[True, False])
    .reset_index(drop=True)
)

# 2) Classificação e probabilidades de grupo.
rank_columns = ['group', 'team', 'avg_position', 'avg_points', 'avg_goal_diff', 'top2_prob', 'position']
comparison_rankings = ranking[rank_columns].merge(
    ranking_retrained[rank_columns],
    on=['group', 'team'],
    suffixes=('_pre', '_post'),
    validate='one_to_one',
)
for metric in ['avg_position', 'avg_points', 'avg_goal_diff', 'top2_prob', 'position']:
    comparison_rankings[f'{metric}_delta'] = (
        comparison_rankings[f'{metric}_post'] - comparison_rankings[f'{metric}_pre']
    )
comparison_rankings['qualification_pre'] = comparison_rankings['position_pre'] <= 2
comparison_rankings['qualification_post'] = comparison_rankings['position_post'] <= 2
comparison_rankings['qualification_changed'] = (
    comparison_rankings['qualification_pre'] != comparison_rankings['qualification_post']
)
comparison_rankings = comparison_rankings.sort_values(['group', 'position_post']).reset_index(drop=True)

# 3) Somente os 48 jogos que ainda não tinham sido disputados após a primeira rodada.
prediction_fields = [
    'predicted_home_goals', 'predicted_away_goals', 'corners',
    'yellow_cards', 'red_cards', 'winning_team',
]
pre_remaining = group_predictions[
    group_predictions['match_id'].isin(remaining_group_predictions_retrained['match_id'])
][['match_id'] + prediction_fields].rename(
    columns={field: f'{field}_pre' for field in prediction_fields}
)
post_remaining = remaining_group_predictions_retrained[
    ['match_id', 'group', 'home_team', 'away_team', 'date_utc', 'venue'] + prediction_fields
].rename(columns={field: f'{field}_post' for field in prediction_fields})
comparison_remaining_matches = post_remaining.merge(
    pre_remaining, on='match_id', how='left', validate='one_to_one'
)
for metric in ['predicted_home_goals', 'predicted_away_goals', 'corners', 'yellow_cards', 'red_cards']:
    comparison_remaining_matches[f'{metric}_delta'] = (
        comparison_remaining_matches[f'{metric}_post']
        - comparison_remaining_matches[f'{metric}_pre']
    )
comparison_remaining_matches['result_changed'] = (
    comparison_remaining_matches['winning_team_pre']
    != comparison_remaining_matches['winning_team_post']
)
comparison_remaining_matches = comparison_remaining_matches.sort_values('match_id').reset_index(drop=True)

# 4) Mata-mata, comparado por vaga/match_id mesmo quando os participantes mudam.
knockout_fields = [
    'predicted_home_team', 'predicted_away_team',
    'predicted_home_goals', 'predicted_away_goals',
    'corners', 'yellow_cards', 'red_cards', 'match_winner', 'penalties',
]
knockout_pre = knockout_predictions[
    ['match_id'] + knockout_fields
].rename(columns={field: f'{field}_pre' for field in knockout_fields})
knockout_post = knockout_predictions_retrained[
    ['match_id', 'round', 'multiplier', 'date_utc', 'venue', 'slot_home', 'slot_away'] + knockout_fields
].rename(columns={field: f'{field}_post' for field in knockout_fields})
comparison_knockout = knockout_post.merge(
    knockout_pre, on='match_id', how='left', validate='one_to_one'
)
comparison_knockout['winner_team_pre'] = np.where(
    comparison_knockout['match_winner_pre'] == 'home',
    comparison_knockout['predicted_home_team_pre'],
    comparison_knockout['predicted_away_team_pre'],
)
comparison_knockout['winner_team_post'] = np.where(
    comparison_knockout['match_winner_post'] == 'home',
    comparison_knockout['predicted_home_team_post'],
    comparison_knockout['predicted_away_team_post'],
)
comparison_knockout['matchup_changed'] = (
    (comparison_knockout['predicted_home_team_pre'] != comparison_knockout['predicted_home_team_post'])
    | (comparison_knockout['predicted_away_team_pre'] != comparison_knockout['predicted_away_team_post'])
)
comparison_knockout['winner_changed'] = (
    comparison_knockout['winner_team_pre'] != comparison_knockout['winner_team_post']
)
comparison_knockout['penalties_changed'] = (
    comparison_knockout['penalties_pre'] != comparison_knockout['penalties_post']
)
for metric in ['predicted_home_goals', 'predicted_away_goals', 'corners', 'yellow_cards', 'red_cards']:
    comparison_knockout[f'{metric}_delta'] = (
        comparison_knockout[f'{metric}_post'] - comparison_knockout[f'{metric}_pre']
    )
comparison_knockout = comparison_knockout.sort_values('match_id').reset_index(drop=True)

# Nomes definitivos das seleções que ocuparam as vagas provisórias.
DISPLAY_NAME_MAP = {
    'UEFA Playoff A': 'Bosnia and Herzegovina',
    'UEFA Playoff B': 'Sweden',
    'UEFA Playoff C': 'Turkey',
    'UEFA Playoff D': 'Czech Republic',
    'FIFA Playoff 1': 'DR Congo',
    'FIFA Playoff 2': 'Iraq',
}

comparison_elo['team'] = comparison_elo['team'].replace(DISPLAY_NAME_MAP)
comparison_rankings['team'] = comparison_rankings['team'].replace(DISPLAY_NAME_MAP)
for column in ['home_team', 'away_team']:
    comparison_remaining_matches[column] = comparison_remaining_matches[column].replace(DISPLAY_NAME_MAP)
for column in [
    'predicted_home_team_pre', 'predicted_away_team_pre',
    'predicted_home_team_post', 'predicted_away_team_post',
    'winner_team_pre', 'winner_team_post',
]:
    comparison_knockout[column] = comparison_knockout[column].replace(DISPLAY_NAME_MAP)

# 5) Resumo de alto nível usado também nos cartões do dashboard.
largest_gain = comparison_elo.loc[comparison_elo['elo_delta'].idxmax()]
largest_loss = comparison_elo.loc[comparison_elo['elo_delta'].idxmin()]
comparison_summary = pd.DataFrame([{
    'first_round_matches_used': len(first_round_actual),
    'remaining_matches_compared': len(comparison_remaining_matches),
    'teams_compared': len(comparison_rankings),
    'teams_with_position_change': int((comparison_rankings['position_delta'] != 0).sum()),
    'teams_with_qualification_change': int(comparison_rankings['qualification_changed'].sum()),
    'remaining_results_changed': int(comparison_remaining_matches['result_changed'].sum()),
    'knockout_matchups_changed': int(comparison_knockout['matchup_changed'].sum()),
    'knockout_winners_changed': int(comparison_knockout['winner_changed'].sum()),
    'champion_pre': completed[104]['winner'],
    'champion_post': completed_retrained[104]['winner'],
    'largest_elo_gain_team': largest_gain['team'],
    'largest_elo_gain': largest_gain['elo_delta'],
    'largest_elo_loss_team': largest_loss['team'],
    'largest_elo_loss': largest_loss['elo_delta'],
}])

exports = {
    'comparison_summary.csv': comparison_summary,
    'comparison_elo.csv': comparison_elo,
    'comparison_rankings.csv': comparison_rankings,
    'comparison_remaining_matches.csv': comparison_remaining_matches,
    'comparison_knockout.csv': comparison_knockout,
}
for filename, frame in exports.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False, encoding='utf-8-sig')

display(comparison_summary, comparison_elo, comparison_rankings)


,first_round_matches_used,remaining_matches_compared,teams_compared,teams_with_position_change,teams_with_qualification_change,remaining_results_changed,knockout_matchups_changed,knockout_winners_changed,champion_pre,champion_post,largest_elo_gain_team,largest_elo_gain,largest_elo_loss_team,largest_elo_loss
0,24,48,48,22,6,2,24,17,Argentina,Argentina,Sweden,38.023536,Tunisia,-38.268636


,group,team,elo_pre,elo_post,elo_delta
0,A,Mexico,1718.575516,1731.237513,12.661996
1,A,South Korea,1714.058931,1727.086155,13.027225
2,A,South Africa,1606.753037,1593.648201,-13.104836
3,A,Czech Republic,1593.690298,1580.230501,-13.459797
4,B,Switzerland,1687.167999,1678.688358,-8.479641
5,B,Canada,1671.942022,1659.924555,-12.017467
6,B,Qatar,1535.968847,1544.112892,8.144045
7,B,Bosnia and Herzegovina,1498.281125,1509.924237,11.643112
8,C,Morocco,1800.728980,1797.559288,-3.169691
9,C,Brazil,1752.395550,1754.830337,2.434787


,group,team,avg_position_pre,avg_points_pre,avg_goal_diff_pre,top2_prob_pre,position_pre,avg_position_post,avg_points_post,avg_goal_diff_post,top2_prob_post,position_post,avg_position_delta,avg_points_delta,avg_goal_diff_delta,top2_prob_delta,position_delta,qualification_pre,qualification_post,qualification_changed
0,A,Mexico,1.8010,5.8105,2.6745,0.8065,1,1.5070,6.8075,3.7420,0.9455,1,-0.2940,0.9970,1.0675,0.1390,0,True,True,False
1,A,South Korea,1.8140,5.7145,2.5170,0.7950,2,1.6645,6.6410,2.5035,0.9095,2,-0.1495,0.9265,-0.0135,0.1145,0,True,True,False
2,A,Czech Republic,3.2445,2.5070,-2.7615,0.1770,4,3.3755,1.7670,-2.8045,0.0785,3,0.1310,-0.7400,-0.0430,-0.0985,-1,False,False,False
3,A,South Africa,3.1405,2.6665,-2.4300,0.2215,3,3.4530,1.9305,-3.4410,0.0665,4,0.3125,-0.7360,-1.0110,-0.1550,1,False,False,False
4,B,Switzerland,1.5990,6.4245,3.9270,0.8845,1,1.7570,4.9930,2.1025,0.7885,1,0.1580,-1.4315,-1.8245,-0.0960,0,True,True,False
5,B,Canada,1.7465,6.0510,3.0870,0.8630,2,2.0680,4.4150,1.0435,0.6675,2,0.3215,-1.6360,-2.0435,-0.1955,0,True,True,False
6,B,Qatar,3.1805,2.5395,-2.8275,0.1600,3,2.8640,3.2615,-0.8775,0.3515,3,-0.3165,0.7220,1.9500,0.1915,0,False,False,False
7,B,Bosnia and Herzegovina,3.4740,1.8315,-4.1865,0.0925,4,3.3110,2.4595,-2.2685,0.1925,4,-0.1630,0.6280,1.9180,0.1000,0,False,False,False
8,C,Morocco,1.4225,6.9465,4.9975,0.9480,1,1.5260,6.1620,4.3110,0.9190,1,0.1035,-0.7845,-0.6865,-0.0290,0,True,True,False
9,C,Brazil,1.8630,5.8090,2.8080,0.8375,2,1.8185,5.7305,3.2670,0.8475,2,-0.0445,-0.0785,0.4590,0.0100,0,True,True,False


In [15]:
# Gráficos executivos.
elo_movers = pd.concat([
    comparison_elo.nsmallest(8, 'elo_delta'),
    comparison_elo.nlargest(8, 'elo_delta'),
]).drop_duplicates('team').sort_values('elo_delta')
fig_elo = px.bar(
    elo_movers, x='elo_delta', y='team', color='elo_delta', orientation='h',
    color_continuous_scale='RdYlGn', color_continuous_midpoint=0,
    title='Maiores mudanças de Elo após a primeira rodada',
    labels={'elo_delta': 'Variação de Elo', 'team': 'Seleção'},
)
fig_elo.update_layout(coloraxis_showscale=False, height=520, margin=dict(l=20, r=20, t=60, b=20))

fig_probability = px.scatter(
    comparison_rankings,
    x='top2_prob_pre', y='top2_prob_post', color='group', text='team',
    hover_data=['position_pre', 'position_post', 'top2_prob_delta'],
    title='Probabilidade de terminar no Top 2: pré-Copa × pós-1ª rodada',
    labels={'top2_prob_pre': 'Probabilidade pré-Copa', 'top2_prob_post': 'Probabilidade pós-retreino'},
)
fig_probability.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode='lines', name='Sem mudança',
    line=dict(color='#64748b', dash='dash'),
))
fig_probability.update_traces(textposition='top center', selector=dict(mode='markers+text'))
fig_probability.update_layout(height=620, margin=dict(l=20, r=20, t=60, b=20))

fig_positions = px.bar(
    comparison_rankings.sort_values(['group', 'position_delta']),
    x='team', y='position_delta', color='group',
    hover_data=['position_pre', 'position_post', 'top2_prob_delta'],
    title='Mudança da posição prevista no grupo',
    labels={'position_delta': 'Posição pós − posição pré', 'team': 'Seleção'},
)
fig_positions.add_hline(y=0, line_color='#64748b')
fig_positions.update_layout(height=560, margin=dict(l=20, r=20, t=60, b=100))

transition_counts = (
    comparison_remaining_matches
    .groupby(['winning_team_pre', 'winning_team_post'])
    .size().reset_index(name='count')
)
outcomes = ['home', 'draw', 'away']
labels = [f'Pré: {value}' for value in outcomes] + [f'Pós: {value}' for value in outcomes]
fig_transitions = go.Figure(go.Sankey(
    node=dict(label=labels, pad=20, color=['#2563eb', '#94a3b8', '#ef4444'] * 2),
    link=dict(
        source=[outcomes.index(row.winning_team_pre) for row in transition_counts.itertuples()],
        target=[3 + outcomes.index(row.winning_team_post) for row in transition_counts.itertuples()],
        value=transition_counts['count'].tolist(),
    ),
))
fig_transitions.update_layout(title='Transição dos resultados previstos nos 48 jogos restantes', height=470)

summary = comparison_summary.iloc[0].to_dict()
groups_filter = sorted(comparison_rankings['group'].unique())
template = Template(r'''<!doctype html>
<html lang="pt-BR">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Impacto do retreino — Copa do Mundo 2026</title>
<style>
:root{--bg:#f1f5f9;--card:#fff;--ink:#0f172a;--muted:#64748b;--accent:#2563eb;--line:#e2e8f0}
*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font-family:Inter,Segoe UI,Arial,sans-serif}
.wrap{max-width:1440px;margin:auto;padding:28px}.hero{background:linear-gradient(135deg,#0f172a,#1d4ed8);color:white;padding:32px;border-radius:18px;margin-bottom:22px}
.hero h1{margin:0 0 8px;font-size:32px}.hero p{margin:0;color:#dbeafe}.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:14px;margin:20px 0}
.card,.panel{background:var(--card);border:1px solid var(--line);border-radius:14px;box-shadow:0 5px 18px #0f172a0d}.card{padding:18px}.value{font-size:29px;font-weight:800;color:var(--accent)}.label{font-size:13px;color:var(--muted);margin-top:5px}
.grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:18px}.panel{padding:16px;margin-bottom:18px}.full{grid-column:1/-1}.panel h2{margin:5px 8px 12px}
.filters{display:flex;gap:12px;flex-wrap:wrap;margin:18px 0}.filters select,.filters input{padding:10px 12px;border:1px solid #cbd5e1;border-radius:9px;background:white;min-width:180px}
.table-wrap{overflow:auto;max-height:570px}table{border-collapse:collapse;width:100%;font-size:13px}th{position:sticky;top:0;background:#e2e8f0;z-index:1}th,td{padding:9px;border-bottom:1px solid var(--line);text-align:left;white-space:nowrap}.changed{background:#fff7ed}
.links a{display:inline-block;margin:4px 8px 4px 0;padding:8px 11px;background:#dbeafe;color:#1e40af;text-decoration:none;border-radius:8px}@media(max-width:900px){.grid{grid-template-columns:1fr}.wrap{padding:14px}}
</style>
</head>
<body><main class="wrap">
<section class="hero"><h1>Impacto do retreino</h1><p>Visão pré-Copa × pós-1ª rodada — Copa do Mundo 2026</p></section>
<section class="cards">
 <div class="card"><div class="value">{{ summary.remaining_results_changed }}</div><div class="label">resultados alterados em 48 jogos</div></div>
 <div class="card"><div class="value">{{ summary.teams_with_qualification_change }}</div><div class="label">seleções mudaram status de classificação</div></div>
 <div class="card"><div class="value">{{ summary.knockout_matchups_changed }}</div><div class="label">confrontos do mata-mata alterados</div></div>
 <div class="card"><div class="value">{{ summary.champion_pre }} → {{ summary.champion_post }}</div><div class="label">campeã prevista pré × pós</div></div>
</section>
<section class="grid">
 <div class="panel">{{ elo_chart }}</div><div class="panel">{{ transition_chart }}</div>
 <div class="panel full">{{ probability_chart }}</div><div class="panel full">{{ positions_chart }}</div>
</section>
<section class="panel"><h2>Explorar detalhes</h2><div class="filters"><select id="group-filter" onchange="filterTables()"><option value="">Todos os grupos</option>{% for group in groups %}<option value="{{ group }}">Grupo {{ group }}</option>{% endfor %}</select><input id="team-filter" oninput="filterTables()" placeholder="Filtrar por seleção"></div>
<h3>Jogos da fase de grupos que mudaram</h3><div class="table-wrap"><table><thead><tr><th>ID</th><th>Grupo</th><th>Confronto</th><th>Placar pré</th><th>Placar pós</th><th>Resultado pré</th><th>Resultado pós</th></tr></thead><tbody>{% for row in changed_matches %}<tr data-group="{{ row.group }}" class="changed"><td>{{ row.match_id }}</td><td>{{ row.group }}</td><td>{{ row.home_team }} × {{ row.away_team }}</td><td>{{ row.predicted_home_goals_pre }}–{{ row.predicted_away_goals_pre }}</td><td>{{ row.predicted_home_goals_post }}–{{ row.predicted_away_goals_post }}</td><td>{{ row.winning_team_pre }}</td><td>{{ row.winning_team_post }}</td></tr>{% endfor %}</tbody></table></div>
<h3>Ranking e classificação</h3><div class="table-wrap"><table><thead><tr><th>Grupo</th><th>Seleção</th><th>Posição pré</th><th>Posição pós</th><th>Δ posição</th><th>Top 2 pré</th><th>Top 2 pós</th><th>Δ Top 2</th></tr></thead><tbody>{% for row in rankings %}<tr data-group="{{ row.group }}" class="{% if row.qualification_changed %}changed{% endif %}"><td>{{ row.group }}</td><td>{{ row.team }}</td><td>{{ row.position_pre }}</td><td>{{ row.position_post }}</td><td>{{ '%+.0f'|format(row.position_delta) }}</td><td>{{ '%.1f%%'|format(row.top2_prob_pre*100) }}</td><td>{{ '%.1f%%'|format(row.top2_prob_post*100) }}</td><td>{{ '%+.1f pp'|format(row.top2_prob_delta*100) }}</td></tr>{% endfor %}</tbody></table></div>
<h3>Todos os jogos restantes da fase de grupos</h3><div class="table-wrap"><table><thead><tr><th>ID</th><th>Grupo</th><th>Confronto</th><th>Placar pré</th><th>Placar pós</th><th>Resultado pré</th><th>Resultado pós</th></tr></thead><tbody>{% for row in matches %}<tr data-group="{{ row.group }}" class="{% if row.result_changed %}changed{% endif %}"><td>{{ row.match_id }}</td><td>{{ row.group }}</td><td>{{ row.home_team }} × {{ row.away_team }}</td><td>{{ row.predicted_home_goals_pre }}–{{ row.predicted_away_goals_pre }}</td><td>{{ row.predicted_home_goals_post }}–{{ row.predicted_away_goals_post }}</td><td>{{ row.winning_team_pre }}</td><td>{{ row.winning_team_post }}</td></tr>{% endfor %}</tbody></table></div>
<h3>Mata-mata</h3><div class="table-wrap"><table><thead><tr><th>ID</th><th>Fase</th><th>Confronto pré</th><th>Placar pré</th><th>Confronto pós</th><th>Placar pós</th><th>Vencedor pré</th><th>Vencedor pós</th></tr></thead><tbody>{% for row in knockout %}<tr class="{% if row.matchup_changed or row.winner_changed %}changed{% endif %}"><td>{{ row.match_id }}</td><td>{{ row.round }}</td><td>{{ row.predicted_home_team_pre }} × {{ row.predicted_away_team_pre }}</td><td>{{ row.predicted_home_goals_pre }}–{{ row.predicted_away_goals_pre }}</td><td>{{ row.predicted_home_team_post }} × {{ row.predicted_away_team_post }}</td><td>{{ row.predicted_home_goals_post }}–{{ row.predicted_away_goals_post }}</td><td>{{ row.winner_team_pre }}</td><td>{{ row.winner_team_post }}</td></tr>{% endfor %}</tbody></table></div>
</section>
<section class="panel links"><h2>Dados CSV</h2>{% for filename in files %}<a href="{{ filename }}">{{ filename }}</a>{% endfor %}</section>
</main><script>
function filterTables(){const g=document.getElementById('group-filter').value;const q=document.getElementById('team-filter').value.toLowerCase();document.querySelectorAll('tbody tr').forEach(row=>{const group=row.dataset.group||'';const groupOk=!g||!group||group===g;const teamOk=!q||row.innerText.toLowerCase().includes(q);row.style.display=groupOk&&teamOk?'':'none';});}
</script></body></html>''')

dashboard_html = template.render(
    summary=summary,
    groups=groups_filter,
    rankings=comparison_rankings.to_dict('records'),
    changed_matches=comparison_remaining_matches[comparison_remaining_matches['result_changed']].to_dict('records'),
    matches=comparison_remaining_matches.to_dict('records'),
    knockout=comparison_knockout.to_dict('records'),
    files=list(exports),
    elo_chart=pio.to_html(fig_elo, full_html=False, include_plotlyjs='inline'),
    transition_chart=pio.to_html(fig_transitions, full_html=False, include_plotlyjs=False),
    probability_chart=pio.to_html(fig_probability, full_html=False, include_plotlyjs=False),
    positions_chart=pio.to_html(fig_positions, full_html=False, include_plotlyjs=False),
)

dashboard_path = OUTPUT_DIR / 'retrain_comparison.html'
dashboard_path.write_text(dashboard_html, encoding='utf-8')

# Critérios de aceitação dos dados e do HTML.
assert len(comparison_elo) == 48 and comparison_elo['team'].is_unique
assert len(comparison_rankings) == 48 and not comparison_rankings.duplicated(['group', 'team']).any()
assert len(comparison_remaining_matches) == 48 and comparison_remaining_matches['match_id'].is_unique
assert len(comparison_knockout) == 32 and comparison_knockout['match_id'].is_unique
assert np.allclose(comparison_elo['elo_delta'], comparison_elo['elo_post'] - comparison_elo['elo_pre'])
assert np.allclose(comparison_rankings['top2_prob_delta'], comparison_rankings['top2_prob_post'] - comparison_rankings['top2_prob_pre'])
assert np.allclose(comparison_rankings['position_delta'], comparison_rankings['position_post'] - comparison_rankings['position_pre'])
for frame, columns in [
    (comparison_elo, ['group', 'team', 'elo_pre', 'elo_post', 'elo_delta']),
    (comparison_rankings, ['group', 'team', 'position_pre', 'position_post', 'top2_prob_pre', 'top2_prob_post']),
    (comparison_remaining_matches, ['match_id', 'group', 'home_team', 'away_team', 'winning_team_pre', 'winning_team_post']),
    (comparison_knockout, ['match_id', 'round', 'predicted_home_team_pre', 'predicted_home_team_post', 'winner_team_pre', 'winner_team_post']),
]:
    assert not frame[columns].isna().any().any()
assert '<script src=' not in dashboard_html.lower()
assert 'src="https://cdn.plot.ly' not in dashboard_html.lower()
assert dashboard_path.exists() and dashboard_path.stat().st_size > 1_000_000

print(f'CSVs gravados em: {OUTPUT_DIR.resolve()}')
print(f'Dashboard autocontido: {dashboard_path.resolve()}')


CSVs gravados em: C:\Users\lucas\repo\DataScience\Prevendo-copa-do-mundo\outputs
Dashboard autocontido: C:\Users\lucas\repo\DataScience\Prevendo-copa-do-mundo\outputs\retrain_comparison.html


## 9. Avaliação da primeira rodada

Compara as previsões pré-Copa com os 24 resultados reais da primeira rodada. O dashboard separa **acerto do resultado** (casa, empate ou visitante) de **placar exato** e apresenta as duas métricas de forma consolidada e diária.


In [16]:
FIRST_ROUND_OUTPUT = Path('outputs')
FIRST_ROUND_OUTPUT.mkdir(parents=True, exist_ok=True)

first_round_ids = set(first_round_actual['match_id'])
pre_first_round = group_predictions[
    group_predictions['match_id'].isin(first_round_ids)
].copy()
assert len(pre_first_round) == 24

def outcome_first_score(home, away, max_goals=8):
    """Escolhe o resultado agregado mais provável e depois a moda de placar dentro dele."""
    lambda_home, lambda_away = lambdas(home, away)
    goals = np.arange(max_goals + 1)
    matrix = np.outer(
        poisson.pmf(goals, lambda_home),
        poisson.pmf(goals, lambda_away),
    )
    total_mass = matrix.sum()
    probabilities = {
        'home': float(np.tril(matrix, k=-1).sum() / total_mass),
        'draw': float(np.trace(matrix) / total_mass),
        'away': float(np.triu(matrix, k=1).sum() / total_mass),
    }
    selected_result = max(probabilities, key=probabilities.get)
    if selected_result == 'home':
        mask = np.greater.outer(goals, goals)
    elif selected_result == 'away':
        mask = np.less.outer(goals, goals)
    else:
        mask = np.equal.outer(goals, goals)
    selected_matrix = np.where(mask, matrix, -1.0)
    home_goals, away_goals = np.unravel_index(np.argmax(selected_matrix), selected_matrix.shape)
    return int(home_goals), int(away_goals), selected_result, probabilities

# O histórico usa os nomes definitivos e sua própria data oficial do jogo.
actual_first_round_lookup = {
    (row.home_team, row.away_team): {
        'actual_date': pd.Timestamp(row.date).date(),
        'actual_home_goals': int(row.home_score),
        'actual_away_goals': int(row.away_score),
    }
    for row in first_round_results.itertuples(index=False)
}

evaluation_rows = []
for row in pre_first_round.itertuples(index=False):
    key = (norm(row.home_team), norm(row.away_team))
    if key not in actual_first_round_lookup:
        raise KeyError(f'Resultado real não encontrado para {row.home_team} × {row.away_team}')
    actual = actual_first_round_lookup[key]
    score_mode_result = (
        'home' if row.predicted_home_goals > row.predicted_away_goals
        else 'away' if row.predicted_away_goals > row.predicted_home_goals
        else 'draw'
    )
    actual_result = (
        'home' if actual['actual_home_goals'] > actual['actual_away_goals']
        else 'away' if actual['actual_away_goals'] > actual['actual_home_goals']
        else 'draw'
    )
    outcome_home, outcome_away, outcome_result, probabilities = outcome_first_score(
        row.home_team, row.away_team
    )
    score_mode_result_correct = score_mode_result == actual_result
    score_mode_exact = (
        int(row.predicted_home_goals) == actual['actual_home_goals']
        and int(row.predicted_away_goals) == actual['actual_away_goals']
    )
    outcome_result_correct = outcome_result == actual_result
    outcome_exact = (
        outcome_home == actual['actual_home_goals']
        and outcome_away == actual['actual_away_goals']
    )
    evaluation_rows.append({
        'match_id': row.match_id,
        'actual_date': actual['actual_date'],
        'group': row.group,
        'home_team': DISPLAY_NAME_MAP.get(row.home_team, row.home_team),
        'away_team': DISPLAY_NAME_MAP.get(row.away_team, row.away_team),
        'score_mode_home_goals': int(row.predicted_home_goals),
        'score_mode_away_goals': int(row.predicted_away_goals),
        'score_mode_result': score_mode_result,
        'score_mode_result_correct': score_mode_result_correct,
        'score_mode_exact_score': score_mode_exact,
        'outcome_first_home_goals': outcome_home,
        'outcome_first_away_goals': outcome_away,
        'outcome_first_result': outcome_result,
        'outcome_first_result_correct': outcome_result_correct,
        'outcome_first_exact_score': outcome_exact,
        'probability_home': probabilities['home'],
        'probability_draw': probabilities['draw'],
        'probability_away': probabilities['away'],
        'actual_home_goals': actual['actual_home_goals'],
        'actual_away_goals': actual['actual_away_goals'],
        'actual_result': actual_result,
        # Compatibilidade com a primeira versão do CSV.
        'predicted_home_goals': int(row.predicted_home_goals),
        'predicted_away_goals': int(row.predicted_away_goals),
        'predicted_result': score_mode_result,
        'result_correct': score_mode_result_correct,
        'exact_score': score_mode_exact,
    })

first_round_evaluation = pd.DataFrame(evaluation_rows).sort_values(
    ['actual_date', 'match_id']
).reset_index(drop=True)

first_round_daily_metrics = (
    first_round_evaluation.groupby('actual_date', as_index=False)
    .agg(
        matches=('match_id', 'size'),
        score_mode_correct_results=('score_mode_result_correct', 'sum'),
        score_mode_exact_scores=('score_mode_exact_score', 'sum'),
        outcome_first_correct_results=('outcome_first_result_correct', 'sum'),
        outcome_first_exact_scores=('outcome_first_exact_score', 'sum'),
    )
)
for technique in ['score_mode', 'outcome_first']:
    first_round_daily_metrics[f'{technique}_result_accuracy_pct'] = (
        100 * first_round_daily_metrics[f'{technique}_correct_results']
        / first_round_daily_metrics['matches']
    )
    first_round_daily_metrics[f'{technique}_exact_score_accuracy_pct'] = (
        100 * first_round_daily_metrics[f'{technique}_exact_scores']
        / first_round_daily_metrics['matches']
    )

first_round_group_metrics = (
    first_round_evaluation.groupby('group', as_index=False)
    .agg(
        matches=('match_id', 'size'),
        score_mode_correct_results=('score_mode_result_correct', 'sum'),
        score_mode_exact_scores=('score_mode_exact_score', 'sum'),
        outcome_first_correct_results=('outcome_first_result_correct', 'sum'),
        outcome_first_exact_scores=('outcome_first_exact_score', 'sum'),
    )
)
for technique in ['score_mode', 'outcome_first']:
    first_round_group_metrics[f'{technique}_result_accuracy_pct'] = (
        100 * first_round_group_metrics[f'{technique}_correct_results']
        / first_round_group_metrics['matches']
    )
    first_round_group_metrics[f'{technique}_exact_score_accuracy_pct'] = (
        100 * first_round_group_metrics[f'{technique}_exact_scores']
        / first_round_group_metrics['matches']
    )

first_round_metrics_summary = pd.DataFrame([{
    'matches': len(first_round_evaluation),
    'score_mode_correct_results': int(first_round_evaluation['score_mode_result_correct'].sum()),
    'score_mode_result_accuracy_pct': 100 * first_round_evaluation['score_mode_result_correct'].mean(),
    'score_mode_exact_scores': int(first_round_evaluation['score_mode_exact_score'].sum()),
    'score_mode_exact_score_accuracy_pct': 100 * first_round_evaluation['score_mode_exact_score'].mean(),
    'outcome_first_correct_results': int(first_round_evaluation['outcome_first_result_correct'].sum()),
    'outcome_first_result_accuracy_pct': 100 * first_round_evaluation['outcome_first_result_correct'].mean(),
    'outcome_first_exact_scores': int(first_round_evaluation['outcome_first_exact_score'].sum()),
    'outcome_first_exact_score_accuracy_pct': 100 * first_round_evaluation['outcome_first_exact_score'].mean(),
}])

first_round_evaluation.to_csv(FIRST_ROUND_OUTPUT / 'first_round_evaluation.csv', index=False, encoding='utf-8-sig')
first_round_daily_metrics.to_csv(FIRST_ROUND_OUTPUT / 'first_round_daily_metrics.csv', index=False, encoding='utf-8-sig')
first_round_metrics_summary.to_csv(FIRST_ROUND_OUTPUT / 'first_round_metrics_summary.csv', index=False, encoding='utf-8-sig')
display(first_round_metrics_summary, first_round_daily_metrics, first_round_evaluation)


,matches,score_mode_correct_results,score_mode_result_accuracy_pct,score_mode_exact_scores,score_mode_exact_score_accuracy_pct,outcome_first_correct_results,outcome_first_result_accuracy_pct,outcome_first_exact_scores,outcome_first_exact_score_accuracy_pct
0,24,10,41.666667,3,12.5,12,50.0,1,4.166667


,actual_date,matches,score_mode_correct_results,score_mode_exact_scores,outcome_first_correct_results,outcome_first_exact_scores,score_mode_result_accuracy_pct,score_mode_exact_score_accuracy_pct,outcome_first_result_accuracy_pct,outcome_first_exact_score_accuracy_pct
0,2026-06-11,2,2,0,2,0,100.0,0.0,100.0,0.0
1,2026-06-12,2,1,0,1,0,50.0,0.0,50.0,0.0
2,2026-06-13,4,1,1,1,0,25.0,25.0,25.0,0.0
3,2026-06-14,4,2,0,2,1,50.0,0.0,50.0,25.0
4,2026-06-15,4,2,2,0,0,50.0,50.0,0.0,0.0
5,2026-06-16,4,1,0,4,0,25.0,0.0,100.0,0.0
6,2026-06-17,4,1,0,2,0,25.0,0.0,50.0,0.0


,match_id,actual_date,group,home_team,away_team,score_mode_home_goals,score_mode_away_goals,score_mode_result,score_mode_result_correct,score_mode_exact_score,...,probability_draw,probability_away,actual_home_goals,actual_away_goals,actual_result,predicted_home_goals,predicted_away_goals,predicted_result,result_correct,exact_score
0,1,2026-06-11,A,Mexico,South Africa,1,0,home,True,False,...,0.206177,0.140197,2,0,home,1,0,home,True,False
1,2,2026-06-11,A,South Korea,Czech Republic,2,0,home,True,False,...,0.198280,0.128387,2,1,home,2,0,home,True,False
2,3,2026-06-12,B,Canada,Bosnia and Herzegovina,2,0,home,False,False,...,0.145337,0.070139,1,1,draw,2,0,home,False,False
3,4,2026-06-12,D,USA,Paraguay,1,0,home,True,False,...,0.245830,0.228896,4,1,home,1,0,home,True,False
4,5,2026-06-13,D,Australia,Turkey,1,1,draw,False,False,...,0.260000,0.303632,2,0,home,1,1,draw,False,False
5,6,2026-06-13,B,Qatar,Switzerland,0,2,away,False,False,...,0.151987,0.755655,1,1,draw,0,2,away,False,False
6,7,2026-06-13,C,Brazil,Morocco,1,1,draw,True,True,...,0.248048,0.486683,1,1,draw,1,1,draw,True,True
7,8,2026-06-13,C,Haiti,Scotland,1,1,draw,False,False,...,0.258966,0.294245,0,1,away,1,1,draw,False,False
8,9,2026-06-14,E,Germany,Curaçao,2,0,home,True,False,...,0.140666,0.066221,7,1,home,2,0,home,True,False
9,10,2026-06-14,F,Netherlands,Japan,1,1,draw,True,False,...,0.254561,0.450586,2,2,draw,1,1,draw,True,False


In [17]:
technique_labels = {
    'score_mode': 'Moda do placar',
    'outcome_first': 'Resultado primeiro',
}

fig_daily_result = go.Figure()
fig_daily_exact = go.Figure()
colors = {'score_mode': '#2563eb', 'outcome_first': '#f97316'}
for technique, label in technique_labels.items():
    fig_daily_result.add_trace(go.Scatter(
        x=first_round_daily_metrics['actual_date'].astype(str),
        y=first_round_daily_metrics[f'{technique}_result_accuracy_pct'],
        mode='lines+markers+text',
        text=first_round_daily_metrics[f'{technique}_result_accuracy_pct'].map(lambda value: f'{value:.0f}%'),
        textposition='top center', name=label,
        line=dict(color=colors[technique], width=3), marker=dict(size=9),
    ))
    fig_daily_exact.add_trace(go.Scatter(
        x=first_round_daily_metrics['actual_date'].astype(str),
        y=first_round_daily_metrics[f'{technique}_exact_score_accuracy_pct'],
        mode='lines+markers+text',
        text=first_round_daily_metrics[f'{technique}_exact_score_accuracy_pct'].map(lambda value: f'{value:.0f}%'),
        textposition='top center', name=label,
        line=dict(color=colors[technique], width=3), marker=dict(size=9),
    ))
for figure, title in [
    (fig_daily_result, 'Resultado correto por dia'),
    (fig_daily_exact, 'Placar exato por dia'),
]:
    figure.update_layout(
        title=title, xaxis_title='Data', yaxis_title='Acerto (%)',
        yaxis=dict(range=[0, 105], ticksuffix='%'), hovermode='x unified',
        height=480, margin=dict(l=30, r=20, t=65, b=40),
    )

summary = first_round_metrics_summary.iloc[0].to_dict()
daily_records = first_round_daily_metrics.copy()
daily_records['actual_date'] = daily_records['actual_date'].astype(str)
matches_records = first_round_evaluation.copy()
matches_records['actual_date'] = matches_records['actual_date'].astype(str)
result_best = max(
    technique_labels,
    key=lambda key: summary[f'{key}_result_accuracy_pct'],
)
exact_best = max(
    technique_labels,
    key=lambda key: summary[f'{key}_exact_score_accuracy_pct'],
)

first_round_template = Template(r'''<!doctype html>
<html lang="pt-BR"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Comparação de técnicas — primeira rodada</title>
<style>
:root{--bg:#f1f5f9;--card:#fff;--ink:#0f172a;--muted:#64748b;--blue:#2563eb;--orange:#f97316;--green:#16a34a;--red:#dc2626;--line:#e2e8f0}*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font-family:Inter,Segoe UI,Arial,sans-serif}.wrap{max-width:1450px;margin:auto;padding:28px}.hero{background:linear-gradient(135deg,#0f172a,#1d4ed8);color:white;padding:32px;border-radius:18px}.hero h1{margin:0 0 8px}.hero p{margin:0;color:#dbeafe}.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:15px;margin:20px 0}.card,.panel{background:white;border:1px solid var(--line);border-radius:14px;box-shadow:0 5px 18px #0f172a0d}.card{padding:20px}.value{font-size:29px;font-weight:800}.blue{color:var(--blue)}.orange{color:var(--orange)}.label{color:var(--muted);font-size:13px;margin-top:5px}.verdict{padding:16px 20px;background:#ecfdf5;border:1px solid #86efac;border-radius:12px;margin-bottom:18px}.grid{display:grid;grid-template-columns:1fr 1fr;gap:18px}.panel{padding:17px;margin-bottom:18px}.filters{display:flex;gap:12px;flex-wrap:wrap;margin:15px 0}.filters select,.filters input{padding:10px;border:1px solid #cbd5e1;border-radius:8px;background:white}.table-wrap{overflow:auto;max-height:700px}table{border-collapse:collapse;width:100%;font-size:12px}th{position:sticky;top:0;background:#e2e8f0;z-index:1}th,td{padding:9px;border-bottom:1px solid var(--line);text-align:left;white-space:nowrap}.yes{color:var(--green);font-weight:700}.no{color:var(--red);font-weight:700}.different{background:#fff7ed}.prob{color:var(--muted)}.links a{display:inline-block;margin-right:8px;padding:8px 11px;border-radius:8px;background:#dbeafe;color:#1e40af;text-decoration:none}@media(max-width:900px){.grid{grid-template-columns:1fr}.wrap{padding:14px}}
</style></head><body><main class="wrap">
<section class="hero"><h1>Moda do placar × resultado primeiro</h1><p>Comparação das duas técnicas nos 24 jogos da primeira rodada</p></section>
<section class="cards">
 <div class="card"><div class="value blue">{{ summary.score_mode_correct_results }}/{{ summary.matches }}</div><div class="label">resultado correto — moda do placar ({{ '%.1f%%'|format(summary.score_mode_result_accuracy_pct) }})</div></div>
 <div class="card"><div class="value orange">{{ summary.outcome_first_correct_results }}/{{ summary.matches }}</div><div class="label">resultado correto — resultado primeiro ({{ '%.1f%%'|format(summary.outcome_first_result_accuracy_pct) }})</div></div>
 <div class="card"><div class="value blue">{{ summary.score_mode_exact_scores }}/{{ summary.matches }}</div><div class="label">placar exato — moda do placar ({{ '%.1f%%'|format(summary.score_mode_exact_score_accuracy_pct) }})</div></div>
 <div class="card"><div class="value orange">{{ summary.outcome_first_exact_scores }}/{{ summary.matches }}</div><div class="label">placar exato — resultado primeiro ({{ '%.1f%%'|format(summary.outcome_first_exact_score_accuracy_pct) }})</div></div>
</section>
<section class="verdict"><strong>Melhor técnica nesta amostra:</strong> resultado correto — {{ result_best }}; placar exato — {{ exact_best }}. A amostra contém somente 24 jogos, portanto diferenças pequenas não devem ser tratadas como conclusão definitiva.</section>
<section class="grid"><div class="panel">{{ result_chart }}</div><div class="panel">{{ exact_chart }}</div></section>
<section class="panel"><h2>Comparação diária</h2><div class="table-wrap"><table><thead><tr><th>Data</th><th>Jogos</th><th>Resultado — moda</th><th>Resultado — primeiro</th><th>Exato — moda</th><th>Exato — primeiro</th></tr></thead><tbody>{% for row in daily %}<tr><td>{{ row.actual_date }}</td><td>{{ row.matches }}</td><td>{{ row.score_mode_correct_results }}/{{ row.matches }} ({{ '%.1f%%'|format(row.score_mode_result_accuracy_pct) }})</td><td>{{ row.outcome_first_correct_results }}/{{ row.matches }} ({{ '%.1f%%'|format(row.outcome_first_result_accuracy_pct) }})</td><td>{{ row.score_mode_exact_scores }}/{{ row.matches }} ({{ '%.1f%%'|format(row.score_mode_exact_score_accuracy_pct) }})</td><td>{{ row.outcome_first_exact_scores }}/{{ row.matches }} ({{ '%.1f%%'|format(row.outcome_first_exact_score_accuracy_pct) }})</td></tr>{% endfor %}</tbody></table></div></section>
<section class="panel"><h2>Partida a partida</h2><div class="filters"><select id="group-filter" onchange="filterRows()"><option value="">Todos os grupos</option>{% for group in groups %}<option value="{{ group }}">Grupo {{ group }}</option>{% endfor %}</select><select id="date-filter" onchange="filterRows()"><option value="">Todas as datas</option>{% for date in dates %}<option value="{{ date }}">{{ date }}</option>{% endfor %}</select><input id="team-filter" oninput="filterRows()" placeholder="Filtrar seleção"></div>
<div class="table-wrap"><table><thead><tr><th>Data</th><th>Grupo</th><th>Confronto</th><th>Moda do placar</th><th>Resultado primeiro</th><th>Prob. casa / empate / fora</th><th>Realidade</th><th>Resultado moda</th><th>Resultado primeiro</th><th>Exato moda</th><th>Exato primeiro</th></tr></thead><tbody>{% for row in matches %}<tr data-group="{{ row.group }}" data-date="{{ row.actual_date }}" class="{% if row.score_mode_result != row.outcome_first_result %}different{% endif %}"><td>{{ row.actual_date }}</td><td>{{ row.group }}</td><td>{{ row.home_team }} × {{ row.away_team }}</td><td>{{ row.score_mode_home_goals }}–{{ row.score_mode_away_goals }}</td><td>{{ row.outcome_first_home_goals }}–{{ row.outcome_first_away_goals }}</td><td class="prob">{{ '%.0f%%'|format(row.probability_home*100) }} / {{ '%.0f%%'|format(row.probability_draw*100) }} / {{ '%.0f%%'|format(row.probability_away*100) }}</td><td>{{ row.actual_home_goals }}–{{ row.actual_away_goals }}</td><td class="{{ 'yes' if row.score_mode_result_correct else 'no' }}">{{ '✓' if row.score_mode_result_correct else '✗' }}</td><td class="{{ 'yes' if row.outcome_first_result_correct else 'no' }}">{{ '✓' if row.outcome_first_result_correct else '✗' }}</td><td class="{{ 'yes' if row.score_mode_exact_score else 'no' }}">{{ '✓' if row.score_mode_exact_score else '✗' }}</td><td class="{{ 'yes' if row.outcome_first_exact_score else 'no' }}">{{ '✓' if row.outcome_first_exact_score else '✗' }}</td></tr>{% endfor %}</tbody></table></div></section>
<section class="panel links"><h2>Dados</h2><a href="first_round_metrics_summary.csv">Resumo CSV</a><a href="first_round_daily_metrics.csv">Métricas diárias CSV</a><a href="first_round_evaluation.csv">Partidas CSV</a></section>
</main><script>function filterRows(){const g=document.getElementById('group-filter').value,d=document.getElementById('date-filter').value,q=document.getElementById('team-filter').value.toLowerCase();document.querySelectorAll('tbody tr[data-group]').forEach(r=>{const ok=(!g||r.dataset.group===g)&&(!d||r.dataset.date===d)&&(!q||r.innerText.toLowerCase().includes(q));r.style.display=ok?'':'none';});}</script></body></html>''')

first_round_html = first_round_template.render(
    summary=summary,
    result_best=technique_labels[result_best],
    exact_best=technique_labels[exact_best],
    daily=daily_records.to_dict('records'),
    matches=matches_records.to_dict('records'),
    groups=sorted(first_round_evaluation['group'].unique()),
    dates=sorted(matches_records['actual_date'].unique()),
    result_chart=pio.to_html(fig_daily_result, full_html=False, include_plotlyjs='inline'),
    exact_chart=pio.to_html(fig_daily_exact, full_html=False, include_plotlyjs=False),
)
first_round_dashboard_path = FIRST_ROUND_OUTPUT / 'first_round_techniques_comparison.html'
first_round_dashboard_path.write_text(first_round_html, encoding='utf-8')

# Relatório isolado da técnica originalmente escolhida: moda do placar.
fig_score_mode_daily = go.Figure()
fig_score_mode_daily.add_trace(go.Scatter(
    x=first_round_daily_metrics['actual_date'].astype(str),
    y=first_round_daily_metrics['score_mode_result_accuracy_pct'],
    mode='lines+markers+text',
    text=first_round_daily_metrics['score_mode_result_accuracy_pct'].map(lambda value: f'{value:.0f}%'),
    textposition='top center', name='Resultado correto',
    line=dict(color='#2563eb', width=3), marker=dict(size=9),
))
fig_score_mode_daily.add_trace(go.Scatter(
    x=first_round_daily_metrics['actual_date'].astype(str),
    y=first_round_daily_metrics['score_mode_exact_score_accuracy_pct'],
    mode='lines+markers+text',
    text=first_round_daily_metrics['score_mode_exact_score_accuracy_pct'].map(lambda value: f'{value:.0f}%'),
    textposition='bottom center', name='Placar exato',
    line=dict(color='#16a34a', width=3), marker=dict(size=9),
))
fig_score_mode_daily.update_layout(
    title='Desempenho diário do modelo original',
    xaxis_title='Data', yaxis_title='Acerto (%)',
    yaxis=dict(range=[0, 105], ticksuffix='%'), hovermode='x unified',
    height=500, margin=dict(l=30, r=20, t=65, b=40),
)

score_mode_template = Template(r'''<!doctype html>
<html lang="pt-BR"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Modelo original — primeira rodada</title>
<style>
:root{--bg:#f1f5f9;--card:#fff;--ink:#0f172a;--muted:#64748b;--blue:#2563eb;--green:#16a34a;--red:#dc2626;--line:#e2e8f0}*{box-sizing:border-box}body{margin:0;background:var(--bg);color:var(--ink);font-family:Inter,Segoe UI,Arial,sans-serif}.wrap{max-width:1380px;margin:auto;padding:28px}.hero{background:linear-gradient(135deg,#0f172a,#1d4ed8);color:white;padding:32px;border-radius:18px}.hero h1{margin:0 0 8px}.hero p{margin:0;color:#dbeafe}.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:15px;margin:20px 0}.card,.panel{background:white;border:1px solid var(--line);border-radius:14px;box-shadow:0 5px 18px #0f172a0d}.card{padding:20px}.value{font-size:32px;font-weight:800}.blue{color:var(--blue)}.green{color:var(--green)}.label{color:var(--muted);font-size:13px;margin-top:5px}.panel{padding:17px;margin-bottom:18px}.filters{display:flex;gap:12px;flex-wrap:wrap;margin:15px 0}.filters select,.filters input{padding:10px;border:1px solid #cbd5e1;border-radius:8px;background:white}.table-wrap{overflow:auto;max-height:700px}table{border-collapse:collapse;width:100%;font-size:13px}th{position:sticky;top:0;background:#e2e8f0;z-index:1}th,td{padding:10px;border-bottom:1px solid var(--line);text-align:left;white-space:nowrap}.yes{color:var(--green);font-weight:700}.no{color:var(--red);font-weight:700}@media(max-width:900px){.wrap{padding:14px}}
</style></head><body><main class="wrap">
<section class="hero"><h1>Modelo original — moda do placar</h1><p>Avaliação isolada das previsões pré-Copa nos 24 jogos da primeira rodada</p></section>
<section class="cards">
 <div class="card"><div class="value">{{ summary.matches|int }}</div><div class="label">partidas avaliadas</div></div>
 <div class="card"><div class="value blue">{{ summary.score_mode_correct_results|int }}/{{ summary.matches|int }}</div><div class="label">resultados corretos — {{ '%.1f%%'|format(summary.score_mode_result_accuracy_pct) }}</div></div>
 <div class="card"><div class="value green">{{ summary.score_mode_exact_scores|int }}/{{ summary.matches|int }}</div><div class="label">placares exatos — {{ '%.1f%%'|format(summary.score_mode_exact_score_accuracy_pct) }}</div></div>
</section>
<section class="panel">{{ daily_chart }}</section>
<section class="panel"><h2>Partida a partida</h2><div class="filters"><select id="group-filter" onchange="filterRows()"><option value="">Todos os grupos</option>{% for group in groups %}<option value="{{ group }}">Grupo {{ group }}</option>{% endfor %}</select><select id="date-filter" onchange="filterRows()"><option value="">Todas as datas</option>{% for date in dates %}<option value="{{ date }}">{{ date }}</option>{% endfor %}</select><input id="team-filter" oninput="filterRows()" placeholder="Filtrar seleção"></div>
<div class="table-wrap"><table><thead><tr><th>Data</th><th>Grupo</th><th>Confronto</th><th>Previsão</th><th>Realidade</th><th>Resultado correto</th><th>Placar exato</th></tr></thead><tbody>{% for row in matches %}<tr data-group="{{ row.group }}" data-date="{{ row.actual_date }}"><td>{{ row.actual_date }}</td><td>{{ row.group }}</td><td>{{ row.home_team }} × {{ row.away_team }}</td><td>{{ row.score_mode_home_goals }}–{{ row.score_mode_away_goals }}</td><td>{{ row.actual_home_goals }}–{{ row.actual_away_goals }}</td><td class="{{ 'yes' if row.score_mode_result_correct else 'no' }}">{{ '✓ Sim' if row.score_mode_result_correct else '✗ Não' }}</td><td class="{{ 'yes' if row.score_mode_exact_score else 'no' }}">{{ '✓ Sim' if row.score_mode_exact_score else '✗ Não' }}</td></tr>{% endfor %}</tbody></table></div></section>
</main><script>function filterRows(){const g=document.getElementById('group-filter').value,d=document.getElementById('date-filter').value,q=document.getElementById('team-filter').value.toLowerCase();document.querySelectorAll('tbody tr[data-group]').forEach(r=>{const ok=(!g||r.dataset.group===g)&&(!d||r.dataset.date===d)&&(!q||r.innerText.toLowerCase().includes(q));r.style.display=ok?'':'none';});}</script></body></html>''')

score_mode_html = score_mode_template.render(
    summary=summary,
    matches=matches_records.to_dict('records'),
    groups=sorted(first_round_evaluation['group'].unique()),
    dates=sorted(matches_records['actual_date'].unique()),
    daily_chart=pio.to_html(fig_score_mode_daily, full_html=False, include_plotlyjs='inline'),
)
score_mode_dashboard_path = FIRST_ROUND_OUTPUT / 'first_round_evaluation.html'
score_mode_dashboard_path.write_text(score_mode_html, encoding='utf-8')
assert '<script src=' not in score_mode_html.lower()
assert score_mode_dashboard_path.exists() and score_mode_dashboard_path.stat().st_size > 1_000_000
print(f'Dashboard isolado do modelo original: {score_mode_dashboard_path.resolve()}')

assert len(first_round_evaluation) == 24 and first_round_evaluation['match_id'].is_unique
for technique in ['score_mode', 'outcome_first']:
    assert first_round_evaluation[f'{technique}_result_correct'].sum() == first_round_metrics_summary.loc[0, f'{technique}_correct_results']
    assert first_round_evaluation[f'{technique}_exact_score'].sum() == first_round_metrics_summary.loc[0, f'{technique}_exact_scores']
assert first_round_daily_metrics['matches'].sum() == 24
assert np.allclose(
    first_round_evaluation[['probability_home', 'probability_draw', 'probability_away']].sum(axis=1),
    1.0,
)
assert '<script src=' not in first_round_html.lower()
assert first_round_dashboard_path.exists() and first_round_dashboard_path.stat().st_size > 1_000_000
print(f'Dashboard comparativo das técnicas: {first_round_dashboard_path.resolve()}')


Dashboard isolado do modelo original: C:\Users\lucas\repo\DataScience\Prevendo-copa-do-mundo\outputs\first_round_evaluation.html
Dashboard comparativo das técnicas: C:\Users\lucas\repo\DataScience\Prevendo-copa-do-mundo\outputs\first_round_techniques_comparison.html
